# Mastering Imperfect Information with Deep Recurrent Q-Networks
## Leduc Hold'em — DRQN vs DQN vs Heuristic vs Random

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/DLPW/blob/master/notebooks/DLPW_colab.ipynb)

**Google Colab Version**

**Runtime**: GPU recommended (Runtime → Change runtime type → GPU)

## 1. Install Dependencies

In [ ]:
!pip install -q rlcard torch matplotlib

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Dependencies installed")
print(f"✓ Device: {device}")
print(f"✓ PyTorch version: {torch.__version__}")

## 2. Setup Project Files

Choose one option below:

### Option A: Clone from GitHub (Recommended)

In [ ]:
import os
import sys

# Clone repository
!git clone https://github.com/YOUR_USERNAME/DLPW.git
%cd DLPW

# Add to Python path
sys.path.insert(0, '/content/DLPW')

print("✓ Repository cloned and path configured")

### Option B: Mount Google Drive

In [ ]:
# Uncomment and run if project is in Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# 
# # Update path to your DLPW folder location in Drive
# project_path = '/content/drive/MyDrive/DLPW'
# %cd {project_path}
# 
# import sys
# sys.path.insert(0, project_path)
# 
# print("✓ Google Drive mounted and path configured")

## 3. Update Config for Colab (Faster Training)

In [ ]:
# Override config for faster Colab training
import config

# Reduce episodes for faster training on Colab
config.NUM_EPISODES_PHASE1 = 5000   # Original: 15000
config.NUM_EPISODES_PHASE2 = 5000   # Original: 15000
config.EVALUATE_EVERY = 500         # Original: 1000
config.EVALUATE_NUM = 500           # Original: 1000
config.NUM_EVAL_HANDS = 1000        # Original: 3000

# Update output paths for Colab
config.OUTPUT_DIR = '/content/outputs'
config.MODEL_DIR = '/content/outputs/models'
config.PLOT_DIR = '/content/outputs/plots'

# Create directories
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.MODEL_DIR, exist_ok=True)
os.makedirs(config.PLOT_DIR, exist_ok=True)

print("✓ Configuration updated for Colab")
print(f"  Phase 1: {config.NUM_EPISODES_PHASE1} episodes")
print(f"  Phase 2: {config.NUM_EPISODES_PHASE2} episodes")
print(f"  Device: {config.DEVICE}")
print(f"  Output: {config.OUTPUT_DIR}")

## 4. Import Modules

In [ ]:
import rlcard
from rlcard.utils import set_seed
from rlcard.agents import RandomAgent, DQNAgent

from config import *
from agents import DRQNAgent, ConservativeHeuristicAgent
from training import CurriculumTrainer
from evaluation import evaluate_agents, compute_action_distribution
from analysis import plot_training_curves, plot_ev_comparison_bar

print("✓ All modules imported successfully")

## 5. Environment Setup

In [ ]:
set_seed(SEED)
env = rlcard.make(ENV_NAME)

raw_shape = env.state_shape[0]
state_shape = raw_shape[0] if isinstance(raw_shape, list) else raw_shape
num_actions = env.num_actions

print(f'✓ Environment: {ENV_NAME}')
print(f'✓ State shape: {state_shape} | Num actions: {num_actions}')
print(f'✓ Device: {DEVICE}')

## 6. Initialize Agents

In [ ]:
# DRQN Agent (with LSTM memory)
agent_drqn = DRQNAgent(
    state_shape=state_shape,
    num_actions=num_actions,
    device=DEVICE,
    hidden_size=HIDDEN_SIZE,
    lr=LEARNING_RATE,
    gamma=GAMMA,
    epsilon_start=EPSILON_START,
    epsilon_min=EPSILON_MIN,
    epsilon_decay=EPSILON_DECAY,
    buffer_capacity=BUFFER_CAPACITY,
    batch_size=BATCH_SIZE,
    min_replay=MIN_REPLAY_SIZE,
    target_update_freq=TARGET_UPDATE_FREQ
)

# DQN Agent (feedforward baseline)
agent_dqn = DQNAgent(
    num_actions=num_actions,
    state_shape=env.state_shape[0],
    mlp_layers=[HIDDEN_SIZE, HIDDEN_SIZE],
    device=DEVICE
)

# Baseline agents
agent_random = RandomAgent(num_actions)
agent_heuristic = ConservativeHeuristicAgent(num_actions)

print("✓ DRQN Agent initialized")
print("✓ DQN Agent initialized")
print("✓ Random Agent initialized")
print("✓ Heuristic Agent initialized")

## 7. Train DRQN (Two-Phase Curriculum)

**Phase 1**: Learn basics vs Random  
**Phase 2**: Learn exploitation vs Heuristic

**Note**: Reduced to 5K episodes per phase for Colab. Increase in config for better results.

In [ ]:
trainer_drqn = CurriculumTrainer(
    env=env,
    agent=agent_drqn,
    opponent_phase1=agent_random,
    opponent_phase2=agent_heuristic
)

drqn_history = trainer_drqn.train()

# Save model
drqn_checkpoint = f'{MODEL_DIR}/drqn_final.pt'
agent_drqn.save_model(drqn_checkpoint)
print(f"\n✓ DRQN training complete. Model saved to {drqn_checkpoint}")

## 8. Train DQN (Baseline Comparison)

In [ ]:
trainer_dqn = CurriculumTrainer(
    env=env,
    agent=agent_dqn,
    opponent_phase1=agent_random,
    opponent_phase2=agent_heuristic
)

dqn_history = trainer_dqn.train()
print("\n✓ DQN training complete")

## 9. Visualize Training Curves

In [ ]:
# DRQN training curves
plot_training_curves(
    ev_history_random=drqn_history['ev_random'],
    ev_history_heuristic=drqn_history['ev_heuristic'],
    loss_history=drqn_history['loss'],
    phase1_episodes=NUM_EPISODES_PHASE1,
    save_path=f'{PLOT_DIR}/drqn_training_curves.png'
)

## 10. Final Evaluation

In [ ]:
print(f"\n{'='*60}")
print(f"FINAL EVALUATION ({NUM_EVAL_HANDS} hands per matchup)")
print(f"{'='*60}")

results = {}

# DRQN evaluations
print('\nDRQN vs ...')
ev_vs_random, traj_drqn_vs_random = evaluate_agents(
    env, agent_drqn, agent_random, NUM_EVAL_HANDS, 'Random'
)
ev_vs_heuristic, traj_drqn_vs_heuristic = evaluate_agents(
    env, agent_drqn, agent_heuristic, NUM_EVAL_HANDS, 'Heuristic'
)
ev_vs_dqn, traj_drqn_vs_dqn = evaluate_agents(
    env, agent_drqn, agent_dqn, NUM_EVAL_HANDS, 'Standard DQN'
)

# DQN evaluations
print('\nDQN vs ...')
dqn_vs_random, traj_dqn_vs_random = evaluate_agents(
    env, agent_dqn, agent_random, NUM_EVAL_HANDS, 'Random'
)
dqn_vs_heuristic, traj_dqn_vs_heuristic = evaluate_agents(
    env, agent_dqn, agent_heuristic, NUM_EVAL_HANDS, 'Heuristic'
)

# Store results
results['DRQN vs Random'] = {
    'ev': ev_vs_random,
    'action_stats': compute_action_distribution(traj_drqn_vs_random, ACTION_NAMES)
}
results['DRQN vs Heuristic'] = {
    'ev': ev_vs_heuristic,
    'action_stats': compute_action_distribution(traj_drqn_vs_heuristic, ACTION_NAMES)
}
results['DRQN vs DQN'] = {
    'ev': ev_vs_dqn,
    'action_stats': compute_action_distribution(traj_drqn_vs_dqn, ACTION_NAMES)
}
results['DQN vs Random'] = {
    'ev': dqn_vs_random
}
results['DQN vs Heuristic'] = {
    'ev': dqn_vs_heuristic
}

## 11. Detailed Analysis

In [ ]:
# Print detailed analysis
print("\n" + "="*60)
print("DETAILED ANALYSIS")
print("="*60)

for matchup, metrics in results.items():
    if 'action_stats' in metrics:
        print(f"\n{matchup}:")
        print(f"  EV: {metrics['ev']:+.3f} chips/hand")
        stats = metrics['action_stats']
        print(f"  Bluff Rate: {stats['bluff_rate']:.1f}% "
              f"({stats['bluff_attempts']}/{stats['total_jack_steps']} Jack decisions)")
        
        # Action distribution
        print(f"  Actions:")
        for action, count in sorted(stats['action_counts'].items(), key=lambda x: -x[1]):
            pct = 100 * count / stats['total_actions'] if stats['total_actions'] else 0
            print(f"    {action}: {pct:.1f}%")

# Final verdict
print("\n" + "="*60)
if ev_vs_dqn > 0:
    print("✓ SUCCESS: DRQN outperforms the memoryless DQN!")
    print(f"  DRQN advantage: {ev_vs_dqn:+.3f} chips/hand")
else:
    print("⚠ NOTE: DQN held its ground — consider more training episodes.")
print("="*60)

## 12. Final Comparison Plot

In [ ]:
# EV comparison bar chart
ev_comparison = {
    'DRQN\nvs Random': results['DRQN vs Random']['ev'],
    'DRQN\nvs Heuristic': results['DRQN vs Heuristic']['ev'],
    'DRQN\nvs DQN': results['DRQN vs DQN']['ev'],
    'DQN\nvs Random': results['DQN vs Random']['ev'],
    'DQN\nvs Heuristic': results['DQN vs Heuristic']['ev'],
}
plot_ev_comparison_bar(
    results=ev_comparison,
    save_path=f'{PLOT_DIR}/ev_comparison.png'
)

## 13. Download Results

In [ ]:
# Zip outputs for download
!zip -r /content/dlpw_outputs.zip {OUTPUT_DIR}

# Download
from google.colab import files
files.download('/content/dlpw_outputs.zip')

print("✓ Results zipped and ready for download")

## 14. Save to Google Drive (Optional)

In [ ]:
# Uncomment to save outputs to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# 
# !cp -r {OUTPUT_DIR} '/content/drive/MyDrive/DLPW_outputs'
# print("✓ Outputs saved to Google Drive")

## Summary

### What We Did
1. ✅ Installed dependencies
2. ✅ Cloned project from GitHub
3. ✅ Imported modular code (agents, models, training, evaluation, analysis)
4. ✅ Trained DRQN (5K episodes/phase)
5. ✅ Trained DQN baseline
6. ✅ Evaluated all matchups
7. ✅ Generated visualizations
8. ✅ Downloaded results

### Key Results
- **DRQN vs DQN**: Check if positive EV (memory advantage)
- **Bluff Rate**: Should be 10-25% (strategic reasoning)
- **Training**: Loss decreased, EV increased

### Next Steps
- Increase episodes to 15K/phase for better results
- Experiment with hyperparameters in config
- Try self-play training
- Extend to larger poker variants